# Interactive Building Damage Labeling Interface

This interactive notebook provides a simple visual audit workspace for inspecting building footprint masks, pre/post-event Sentinel-1 SAR backscatter backdrops, pre/post false-color Sentinel-2 composites, and logging double-blind damage labels.

### Target Schema
- `0` = **INTACT** (visibly structurally sound after disaster landfall)
- `1` = **DAMAGED** (roof collapses, debris debris piles, minor/major structural failure)
- `2` = **DESTROYED** (complete collapse, foundation erasure, building completely removed)
- `U` = **UNCERTAIN** (obscured, missing, or contradictory evidence)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load catalog
catalog_path = '../data/tamil_nadu/chips_pilot/pilot_catalog.csv'
if not os.path.exists(catalog_path):
    # Check absolute fallback if cwd is root
    catalog_path = 'data/tamil_nadu/chips_pilot/pilot_catalog.csv'

catalog_df = pd.read_csv(catalog_path)
print(f"Loaded {len(catalog_df)} pilot samples.")

In [ ]:
def display_building_sample(row_idx):
    row = catalog_df.iloc[row_idx]
    b_id = row['building_id']
    event_id = row['event_id']
    b_dir = row['output_dir']
    if b_dir.startswith('data/'):
        b_dir = '../' + b_dir
        
    # Load metadata
    with open(f"{b_dir}/metadata.json", 'r') as fm:
        meta = json.load(fm)
        
    # Load arrays
    sar_pre = np.load(f"{b_dir}/sar_pre.npy") if os.path.exists(f"{b_dir}/sar_pre.npy") else None
    sar_post = np.load(f"{b_dir}/sar_post.npy") if os.path.exists(f"{b_dir}/sar_post.npy") else None
    opt_pre = np.load(f"{b_dir}/optical_pre.npy") if os.path.exists(f"{b_dir}/optical_pre.npy") else None
    opt_post = np.load(f"{b_dir}/optical_post.npy") if os.path.exists(f"{b_dir}/optical_post.npy") else None
    mask = np.load(f"{b_dir}/building_mask.npy") if os.path.exists(f"{b_dir}/building_mask.npy") else None
    
    # Plot
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    
    # SAR
    if sar_pre is not None:
        axes[0, 0].imshow(sar_pre[0, :, :], cmap='gray')
        axes[0, 0].set_title("SAR PRE (VV)")
    if sar_post is not None:
        axes[0, 1].imshow(sar_post[0, :, :], cmap='gray')
        axes[0, 1].set_title("SAR POST (VV)")
    if sar_pre is not None and sar_post is not None:
        diff = np.abs(sar_post[0, :, :] - sar_pre[0, :, :])
        axes[0, 2].imshow(diff, cmap='hot')
        axes[0, 2].set_title("SAR Change Map")
        
    # Optical RGB
    if opt_pre is not None:
        rgb = np.zeros((32, 32, 3))
        rgb[:, :, 0] = opt_pre[3, :, :] # B8
        rgb[:, :, 1] = opt_pre[2, :, :] # B4
        rgb[:, :, 2] = opt_pre[1, :, :] # B3
        for i in range(3):
            p1, p99 = np.percentile(rgb[:, :, i], [1, 99])
            rgb[:, :, i] = np.clip((rgb[:, :, i] - p1) / (p99 - p1 + 1e-5), 0, 1)
        axes[1, 0].imshow(rgb)
        axes[1, 0].set_title("Optical PRE (False Color)")
    else:
        axes[1, 0].text(0.5, 0.5, "N/A (SAR-only)", ha='center', va='center')
        
    if opt_post is not None:
        rgb = np.zeros((32, 32, 3))
        rgb[:, :, 0] = opt_post[3, :, :]
        rgb[:, :, 1] = opt_post[2, :, :]
        rgb[:, :, 2] = opt_post[1, :, :]
        for i in range(3):
            p1, p99 = np.percentile(rgb[:, :, i], [1, 99])
            rgb[:, :, i] = np.clip((rgb[:, :, i] - p1) / (p99 - p1 + 1e-5), 0, 1)
        axes[1, 1].imshow(rgb)
        axes[1, 1].set_title("Optical POST (False Color)")
    else:
        axes[1, 1].text(0.5, 0.5, "N/A (SAR-only)", ha='center', va='center')
        
    # Mask
    if mask is not None:
        axes[1, 2].imshow(mask, cmap='Blues_r')
        axes[1, 2].set_title("Building Footprint Mask")
        
    for row_ax in axes:
        for ax in row_ax:
            ax.axis('off')
            
    plt.suptitle(f"Sample {row_idx+1}/{len(catalog_df)}: {b_id}\nEvent: {meta['event_name']} | Type: {meta['sample_type']}\nArea: {meta['building_area_m2']:.1f} m2", fontsize=13)
    plt.show()

In [ ]:
# Simple slider interface to browse samples
slider = widgets.IntSlider(min=0, max=len(catalog_df)-1, step=1, value=0, description='Index:')
widgets.interactive(display_building_sample, row_idx=slider)